# Banking Voice Agent — Interactive Demo

Run all 5 journeys interactively against the LangGraph orchestrator.

**Pre-requisites** (run once before this notebook):
```bash
pip install -r requirements.txt
uvicorn src.backend.app:app --port 8000 &   # mock bank API
python -m src.backend.seed                   # seed the database
python -m src.rag build                      # build FAISS index
```

Set `GEMINI_API_KEY` in `.env` or the shell before running.

In [1]:
# Install requirements using the notebook magic so the environment is available to the kernel
!python3 -m venv venv312

# Activate virtualenv in a shell command (use ! to run shell commands from the Python cell)
!source venv312/bin/activate

%pip install -r requirements.txt

# run mock backend and setup (no need to import uvicorn in the notebook)
!uvicorn src.backend.app:app --port 8000 &   # mock bank API
!python -m src.backend.seed                   # seed the database
!python -m src.rag build   
# !python -m src.server                   



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
/Users/Shared/0projects/voiceagent/src/backend/seed.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
Seeded src/backend/bank.db — CUST001 / ACC001 / ₹84,250 balance
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: intfloat/multilingual-e5-small
Indexing 40 chunks from 8 docs…
Saved 40 vectors to src/backend/chroma_db/banking_docs


In [ ]:
# !uvicorn src.backend.app:app --port 8000 

INFO:     Started server process [52829]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [2]:
import json, textwrap, os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))  # make src importable

from IPython.display import display, JSON, Markdown
from src.graph import build_graph, initial_state

graph = build_graph()
print('Graph compiled OK')

Graph compiled OK


In [3]:
def run(user_msg: str, *, customer_id='CUST001', slots=None, pending_action=None):
    """Invoke graph and pretty-print the response + A2UI payload."""
    state = initial_state(customer_id, user_msg, slots=slots, pending_action=pending_action)
    result = graph.invoke(state)

    print(f'Intent:  {result["intent"]}')
    print(f'Done:    {result["done"]}')
    print()
    print('=== Agent response ===')
    print(textwrap.fill(result.get('response_text',''), 80))

    if result.get('a2ui'):
        print('\n=== A2UI payload ===')
        display(JSON(result['a2ui']))

    if result.get('pending_slot'):
        print(f'\nWaiting for slot: {result["pending_slot"]}')

    return result

## UC1 — Balance inquiry

In [11]:
r1 = run('mera balance kya hai?')

Intent:  balance_inquiry
Done:    True

=== Agent response ===
Aapke savings account (ending with 1234) ka balance ₹84250.0 hai.

=== A2UI payload ===


<IPython.core.display.JSON object>

## UC1 — Monthly statement

In [12]:
r2 = run('show me my transactions this month')

Intent:  account_statement
Done:    True

=== Agent response ===
Hello! Here is a summary of your transactions from 2026-06-01 to 2026-06-12:  *
**Total Transactions:** 11 * **Total Spent:** 14598.0 * **Total Received:**
45000.0  Let me know if you need help with anything else!

=== A2UI payload ===


<IPython.core.display.JSON object>

### UC1 — Drill-down (simulate tapping the summary card)

In [ ]:
# Simulate the Flutter action event: user tapped 'show_txn_detail'
r2b = run(
    'show full list',
    slots={**r2.get('slots', {}), 'period': 'this_month'},
    pending_action='show_txn_detail',
)

## UC2 — Open Fixed Deposit (multi-turn slot fill)

In [ ]:
# Turn 1: express intent — no slots yet
r_fd1 = run('I want to open a fixed deposit')
print('pending_slot:', r_fd1.get('pending_slot'))

In [ ]:
# Turn 2: user picks a product
r_fd2 = run(
    'FD001 looks good',
    slots={**r_fd1.get('slots', {}), 'product_id': 'FD001'},
)
print('pending_slot:', r_fd2.get('pending_slot'))

In [ ]:
# Turn 3: user enters amount
r_fd3 = run(
    '50000 rupees',
    slots={**r_fd2.get('slots', {}), 'amount': 50000},
)
print('done:', r_fd3['done'])

## UC4 — Fund Transfer with OTP

In [ ]:
# Turn 1: express intent
r_xf1 = run('I want to send money to Rahul')

In [ ]:
# Turn 2: pick payee, enter amount and rail
r_xf2 = run(
    'PYE001, 2000 via IMPS',
    slots={'payee_id': 'PYE001', 'amount': 2000, 'rail': 'imps'},
)
print('pending_action:', r_xf2.get('pending_action'))

In [ ]:
# Turn 3: OTP (correct OTP is 123456 per seed data)
r_xf3 = run(
    '123456',
    slots={**r_xf2.get('slots', {}), 'otp': '123456'},
    pending_action='await_otp',
)
print('done:', r_xf3['done'])

In [ ]:
# OTP failure path — wrong OTP, graph must NOT advance
r_xf_bad = run(
    '000000',
    slots={**r_xf2.get('slots', {}), 'otp': '000000'},
    pending_action='await_otp',
)
assert r_xf_bad['done'] is False, 'Graph must stay on OTP step for wrong OTP'
print('Correctly stayed on OTP step')

## UC3 — Knowledge RAG

In [8]:
r_rag = run('What documents do I need for KYC?')

Intent:  help_knowledge
Done:    True

=== Agent response ===
Based on the provided documents, you need the following for KYC:  * **PAN
card**: Required for all accounts. * **Aadhaar card**: Accepted as a single
document covering both identity and address.  Source:
`src/backend/docs/kyc_requirements.md`

=== A2UI payload ===


<IPython.core.display.JSON object>

In [ ]:
r_rag2 = run('NEFT vs IMPS kya difference hai?')

## UC5 — Complaint

In [ ]:
# Known self-fixable issue: UPI PIN blocked → suggest fix
r_comp = run(
    'My UPI PIN is blocked, payment not going through',
    slots={'description': 'UPI PIN blocked, payment failed repeatedly'},
)

In [ ]:
# Raise a real complaint when no self-fix applies
r_comp2 = run(
    'I was charged twice for the same bill payment last week',
    slots={'description': 'Duplicate charge on bill payment on 2026-06-05'},
)

## Transaction failure lookup

In [ ]:
r_fail = run(
    'Why did TXN009 fail?',
    slots={'txn_id': 'TXN009'},
)

## Hinglish / multi-lingual

In [ ]:
run('Mujhe apna account statement chahiye is mahine ka')

In [9]:
run('FD kaise kholein? minimum amount kya hai?')

Intent:  help_knowledge
Done:    True

=== Agent response ===
दिए गए दस्तावेज़ों में FD कैसे खोलें ("FD kaise kholein") इसकी जानकारी नहीं दी
गई है।   हालांकि, FD के लिए न्यूनतम राशि (minimum amount) इस प्रकार है: *
**Short-Term FD:** Rs 10,000 * **Standard FD:** Rs 10,000 * **Long-Term FD:** Rs
25,000  **Source:** `src/backend/docs/how_to_open_fd.md`

=== A2UI payload ===


<IPython.core.display.JSON object>

{'customer_id': 'CUST001',
 'user_msg': 'FD kaise kholein? minimum amount kya hai?',
 'intent': 'help_knowledge',
 'slots': {},
 'pending_slot': None,
 'pending_action': None,
 'tool_result': None,
 'response_text': 'दिए गए दस्तावेज़ों में FD कैसे खोलें ("FD kaise kholein") इसकी जानकारी नहीं दी गई है। \n\nहालांकि, FD के लिए न्यूनतम राशि (minimum amount) इस प्रकार है:\n* **Short-Term FD:** Rs 10,000\n* **Standard FD:** Rs 10,000\n* **Long-Term FD:** Rs 25,000\n\n**Source:** `src/backend/docs/how_to_open_fd.md`',
 'a2ui': [{'version': 'v0.9',
   'createSurface': {'surfaceId': 'rag_answer',
    'catalogId': 'https://a2ui.org/specification/v0_9/catalogs/basic/catalog.json'}},
  {'version': 'v0.9',
   'updateComponents': {'surfaceId': 'rag_answer',
    'components': [{'id': 'root', 'component': 'Card', 'child': 'col'},
     {'id': 'col', 'component': 'Column', 'children': ['hdg', 'body', 'src']},
     {'id': 'hdg',
      'component': 'Text',
      'text': 'Knowledge Base',
      'variant': 

## A2UI message inspector

In [10]:
def inspect_a2ui(msgs: list[dict]):
    for i, m in enumerate(msgs):
        mtype = next((k for k in m if k not in ('version',)), '?')
        print(f'[{i}] {mtype}')
        if mtype == 'updateComponents':
            comps = m['updateComponents'].get('components', [])
            for c in comps:
                print(f'     {c.get("id","?")} → {c.get("component","?")}')
        elif mtype == 'updateDataModel':
            udm = m['updateDataModel']
            print(f'     path={udm.get("path")}  value={udm.get("value")}')

r_bal = run('balance check karo')
print()
inspect_a2ui(r_bal.get('a2ui') or [])

Intent:  balance_inquiry
Done:    True

=== Agent response ===
Aapke account ka balance **84250.0 INR** hai.

=== A2UI payload ===


<IPython.core.display.JSON object>


[0] createSurface
[1] updateComponents
     root → Card
     col → Column
     lbl → Text
     amt → Text
     row_act → Row
     btn_stmt → Button
     btn_stmt_lbl → Text
